In [ ]:
import numpy as np

regionsNames    =          ["EUGB", "USOA", "CINA", "JPKT", "ROHL", "SOIN", "SOFL"]
regions         = np.array([    30,     25,     10,     10,     10,     10,     5]) * 1e-2

industriesNames =          ["INMB", "RENG", "BKON", "FINZ", "GUPH", "TECH", "LUXS", "ISTR"]
industriesMin   = np.array([    15,     10,     10,     10,     10,     10,      5,      5]) * 1e-2
industriesMax   = np.array([    20,     20,     15,     15,     15,     15,     10,     10]) * 1e-2
industries      = (industriesMin + industriesMax) / 2
industries      = industries / np.sum(industries)

def fct_printMatrix(A):
  nLin, nCol = A.shape
  lin = 0

  print("     ", end="")
  for elem in regionsNames: print("%5s" % elem, end="")
  print()

  while lin < nLin:
    col = 0
    print("%-6s" % industriesNames[lin], end="")

    while col < nCol:
      print("%4.1f "% A[lin, col], end="")
      col += 1

    print()
    lin += 1
    


matrix = regions[None, :] * industries[:, None]

fct_printMatrix(matrix*100)


In [ ]:
import pandas as pd
import numpy as np

regionsNames    =          ["EUGB", "USOA", "CINA", "JPKT", "ROHL", "SOIN", "SOFL"]
industriesNames =          ["INMB", "RENG", "BKON", "FINZ", "GUPH", "TECH", "LUXS", "ISTR"]

regionsGoalVal    = np.array([    30,     25,     10,     10,     10,     10,     5]) * 1e-2

industriesGoalMin = np.array([    15,     10,     10,     10,     10,     10,      5,      5]) * 1e-2
industriesGoalMax = np.array([    20,     20,     15,     15,     15,     15,     10,     10]) * 1e-2
industriesGoalVal = (industriesGoalMin + industriesGoalMax) / 2
industriesGoalVal = industriesGoalVal / np.sum(industriesGoalVal)

matrixGoalValsRel     = pd.DataFrame(index=industriesNames, columns=regionsNames)
matrixTitleList       = pd.DataFrame(index=industriesNames, columns=regionsNames)
matrixTitleCnt        = pd.DataFrame(index=industriesNames, columns=regionsNames)
matrixTitlePercentage = pd.DataFrame(index=industriesNames, columns=regionsNames)

for idxIndustry, industry in enumerate(industriesNames):
  for idxRegion, region in enumerate(regionsNames):
    matrixGoalValsRel.at[industry, region]     = industriesGoalVal[idxIndustry] * regionsGoalVal[idxRegion]
    matrixTitleList.at[industry, region]       = list()
    matrixTitleCnt.at[industry, region]        = 0
    matrixTitlePercentage.at[industry, region] = 0.0

print(matrixGoalValsRel, end="\n"*2)
print(matrixTitleList, end="\n"*2)

In [ ]:
def fct_stripMarkdownTableLine(inpLine):
  while inpLine[ 0] != "|": inpLine = inpLine[1:]
  while inpLine[-1] != "|": inpLine = inpLine[:-1]

  inpLine = inpLine[1:-1]
  inpArgs = inpLine.split("|")

  for idxArg, inpArg in enumerate(inpArgs):
    inpArgs[idxArg] = inpArg.strip()

  return inpArgs

with open("pfioParam.md", "r") as fidPfio:
  inpPfio = fidPfio.readlines()

while inpPfio[0][0] != "|":
   inpPfio = inpPfio[1:]

while inpPfio[-1][0] != "|":
   inpPfio = inpPfio[:-1]

indxsHeader = fct_stripMarkdownTableLine(inpPfio[0])

zeilen = []
for title in inpPfio[2:]:
    zeilen.append(fct_stripMarkdownTableLine(title))

tabPfio = pd.DataFrame(zeilen, columns=indxsHeader)

print(tabPfio.loc[:, "Kurz":"Typ"])

In [ ]:
for idxLin in range(len(tabPfio)):
  region   = tabPfio.iloc[idxLin].loc["Region"]
  industry = tabPfio.iloc[idxLin].loc["Branche"]
  wkn      = tabPfio.iloc[idxLin].loc["WKN"]
  typ      = tabPfio.iloc[idxLin].loc["Typ"]

  if (region in regionsNames) and (industry in industriesNames):
    matrixTitleList.loc[industry, region].append("%s:%s" % (typ, wkn))
    matrixTitleCnt.loc[industry, region] += 1

tmp = matrixGoalValsRel * 100
tmpStr = tmp.to_string(float_format="{:5.2f}".format)

# print("Proz. Anteile im Portfolio SOLL:\n")
# print(tmp.to_string(float_format="{:5.2f}".format))
# print()

tmp = matrixTitleCnt / np.sum(matrixTitleCnt) * 100
tmpStr = tmp.to_string(float_format="{:5.2f}".format)

# print("Proz. Anteile im Portfolio IST:\n")
# print(tmp.to_string(float_format="{:5.2f}".format))
# print()

tmp    = tmp - matrixGoalValsRel * 100
tmpStr = tmp.to_string(float_format="{:5.2f}".format)

print("Proz. Anteile im Portfolio DIFF:\n")
print(tmpStr)
print()

fuenf_kleinste = tmp.astype(float).stack().nsmallest(10).reset_index()
fuenf_kleinste.columns = ["Zeile", "Spalte", "Wert"]

print(fuenf_kleinste)